# 🔧 Исправление Canonical формата Cash Flow Statement на основе официальной отчетности

**Цель**: Создать правильный canonical формат для истории CF, который соответствует реальным данным из официальной отчетности

**Задача**: 
1. Загрузить данные из официального Cash Flow Statement 2024
2. Сравнить с RAW данными из БД (как данные загрузились в БД до маппинга)
3. Сравнить с CANONICAL данными из БД (как данные стали после маппинга в DataMart)
4. Выявить расхождения и понять, правильно ли препроцессинг собрал отчет о движении денежных средств
5. Создать исправленный canonical формат для 2024 года на основе официальных данных
6. **Обновить CSV файл истории с исправленными данными**
7. **Загрузить исправленные данные в RAW БД через `load_history_csv_to_db`**
8. **Проверить, что маппинг RAW → CANONICAL работает правильно**
9. **Убедиться, что препроцессинг правильно использует canonical данные**

**Процесс загрузки данных**:
```
Официальный отчет → CSV файл истории (RAW формат) → RAW БД → CANONICAL БД (через маппинг) → Препроцессинг → Движок
```

**Примечание**: CSV файлы истории используются DataMart как fallback, но для сравнения мы работаем напрямую с БД (RAW и CANONICAL форматы).

**Важно**: 
- Используйте данные в том же порядке, как в официальном отчете
- Проверьте порядок цифр (могут быть ошибки в масштабе)
- Убедитесь, что все статьи правильно сопоставлены с canonical метриками
- **CSV файл истории должен содержать данные в RAW формате (как в официальном отчете)**
- **Маппинг RAW → CANONICAL происходит автоматически при загрузке через DataMart**

**⭐ Новые метрики breakdown (опциональные)**:
- **CFO**: `change_ar`, `change_inventory`, `change_ap`, `change_other_wc`, `change_taxes_payable`, `change_interest_payable`, `lease_payments_cfo_operating`, `lease_interest_cfo_finance`, `lease_interest_cfo_operating`, `impairment`, `stock_based_comp`, `deferred_tax`, `gain_loss_disposal`, `interest_paid`, `taxes_paid`
- **CFI**: `ppe_disposal_proceeds`, `acquisitions`, `divestitures`, `investments_purchases`, `investments_sales`, `other_investing`
- **CFF**: `debt_issuance`, `debt_repayment`, `revolver_draws`, `revolver_repayments`, `finance_lease_principal`, `dividends_paid`, `share_buybacks`, `equity_issuance`, `other_financing`
- **Cash Bridge**: `cash_beginning`, `net_change_in_cash`, `cash_ending`

Эти метрики опциональны и используются для более детального моделирования, если доступны в исходных данных.

## 🔧 Импорты и настройка

In [ ]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import sys
from IPython.display import display, Markdown, HTML
from typing import Optional
from difflib import SequenceMatcher
import re

# Настройка для графиков в Jupyter
%matplotlib inline

# Определение корня проекта (ПЕРЕД импортом engine)
current_dir = Path.cwd()
if (current_dir / 'engine').exists():
    ROOT = current_dir
elif (current_dir.parent / 'engine').exists():
    ROOT = current_dir.parent
elif (current_dir.parent.parent / 'engine').exists():
    ROOT = current_dir.parent.parent
else:
    nb_path = Path(__file__) if '__file__' in globals() else Path.cwd()
    ROOT = nb_path.parent.parent.parent

print(f"ROOT: {ROOT}")
print(f"Engine exists: {(ROOT / 'engine').exists()}")

# Добавляем ROOT в sys.path ПЕРЕД импортом engine
sys.path.insert(0, str(ROOT))

# Теперь импортируем engine
from engine.database.data_mart import get_data_mart

# Определяем COMPANY автоматически из пути ноутбука
COMPANY = 'us_steel'  # fallback по умолчанию
if 'companies' in current_dir.parts:
    companies_idx = current_dir.parts.index('companies')
    if companies_idx + 1 < len(current_dir.parts):
        COMPANY = current_dir.parts[companies_idx + 1]

croot = ROOT / 'companies' / COMPANY
print(f"Company root: {croot}")

MODEL_VERSION: Optional[str] = None

## 📋 ШАГ 1: Данные из официального Cash Flow Statement 2024

In [ ]:
# === Данные из официального Cash Flow Statement 2024 ===
# Данные из официальной отчетности US Steel (в том же порядке, как в отчете)

official_reporting_2024 = {
    # === OPERATING ACTIVITIES ===
    'Net earnings': 384.0,  # миллионов USD
    'Depreciation, depletion and amortization': 913.0,
    'Asset impairment charges': 19.0,
    'Gain on equity investee transactions': 0.0,
    'Restructuring and other charges': 8.0,
    'Loss on debt extinguishment': 2.0,
    'Pensions and other post-employment benefits': -133.0,  # отрицательное = доход
    'Active employee benefit investments': 65.0,
    'Deferred income taxes': 113.0,
    'Net gain on sale of assets': -5.0,  # отрицательное = доход
    'Equity investees earnings, net of distributions received': -3.0,  # отрицательное = доход
    'Changes in: Current receivables': 174.0,
    'Inventories': -71.0,
    'Current accounts payable and accrued expenses': -285.0,
    'Income taxes receivable/payable': -126.0,
    'All other, net': -136.0,
    'Net cash provided by operating activities': 919.0,
    
    # === INVESTING ACTIVITIES ===
    'Capital expenditures': -2287.0,
    'Proceeds from cost reimbursement government grants': 0.0,
    'Proceeds from sale of assets': 5.0,
    'Proceeds from sale of ownership interests in equity investees': 0.0,
    'Other investing activities': 6.0,
    'Net cash used in investing activities': -2276.0,
    
    # === FINANCING ACTIVITIES ===
    'Issuance of long-term debt, net of financing costs': 0.0,
    'Repayment of long-term debt': -128.0,
    'Common stock repurchased': 0.0,
    'Proceeds from government incentives': 0.0,
    'Other financing activities': -71.0,
    'Net cash used in financing activities': -199.0,
    
    # === OTHER ===
    'Effect of exchange rate changes on cash': -19.0,
    'Net (decrease) increase in cash, cash equivalents and restricted cash': -1575.0,
    'Cash, cash equivalents and restricted cash at beginning of year': 2988.0,
    'Cash, cash equivalents and restricted cash at end of year': 1413.0,
}

# Конвертируем из миллионов в доллары
official_reporting_2024_converted = {}
for k, v in official_reporting_2024.items():
    official_reporting_2024_converted[k] = v * 1_000_000

official_reporting_2024 = official_reporting_2024_converted

print(f"✅ Загружено {len(official_reporting_2024)} статей из официального Cash Flow Statement 2024")

# Создаём таблицу для отображения
official_df = pd.DataFrame([
    {
        '№': i + 1,
        'Статья (официальный отчет)': item,
        'Значение (млн USD)': value / 1_000_000,
        'Значение (USD)': value,
    }
    for i, (item, value) in enumerate(official_reporting_2024.items())
])

display(Markdown("### 📊 Таблица данных из официального Cash Flow Statement 2024"))
display(official_df.style.format({
    'Значение (млн USD)': lambda x: f"${x:,.2f}" if abs(x) > 0.01 else "$0.00",
    'Значение (USD)': lambda x: f"${x:,.0f}"
}).set_table_styles([
    {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
]).set_properties(**{'font-size': '9px'}))

print(f"\n📋 Всего статей: {len(official_reporting_2024)}")
print(f"📊 Проверка связки: Net earnings + Adjustments + WC Changes = CFO")
net_earnings = official_reporting_2024.get('Net earnings', 0)
cfo = official_reporting_2024.get('Net cash provided by operating activities', 0)
print(f"   Net earnings: ${net_earnings/1_000_000:,.0f}M")
print(f"   CFO: ${cfo/1_000_000:,.0f}M")

## 📄 ШАГ 2: Загрузка данных из файла истории (опционально)

**Примечание**: Этот шаг опционален. CSV файл (`cf_history_us_steel.csv`) - это результат конвертации Excel файла через `excel_loader.py`.

**Процесс загрузки данных**:
1. **Excel файл** → `excel_loader.py` → **CSV файл** (без маппинга, как есть)
2. **CSV файл** → `load_history_csv_to_db` → **RAW БД** (загружается как есть)
3. **RAW БД** → `normalize_to_canonical` + `excel_loader.yaml` → **CANONICAL БД** (применяется маппинг)

CSV файлы используются DataMart как fallback, но основная работа идет с данными из БД (ШАГ 3).

In [ ]:
# === Загрузка данных из файла истории CF ===

hist_file = croot / "history" / "cf_history_us_steel.csv"

if not hist_file.exists():
    print(f"❌ Файл истории не найден: {hist_file}")
    hist_2024_raw = {}
else:
    hist_df = pd.read_csv(hist_file)
    print(f"✅ Файл истории загружен: {hist_file.name}")
    print(f"   Структура: {hist_df.shape[0]} строк, {hist_df.shape[1]} колонок")
    
    # Получаем данные 2024 из файла истории
    hist_2024_raw = {}
    if '2024' in hist_df.columns:
        for _, row in hist_df.iterrows():
            metric = row['metric']
            value = pd.to_numeric(row['2024'], errors='coerce')
            if pd.notna(value):
                hist_2024_raw[metric] = float(value)
        
        print(f"\n✅ Загружено {len(hist_2024_raw)} статей из файла истории для 2024 года")
    else:
        print(f"❌ Колонка '2024' не найдена в файле истории")
        print(f"   Доступные колонки: {list(hist_df.columns)}")

## 🗄️ ШАГ 3: Загрузка данных из БД (canonical формат)

In [ ]:
# === Загрузка данных из БД: RAW (до маппинга) и CANONICAL (после маппинга) ===

mart = get_data_mart(ROOT, COMPANY)

# RAW данные - как они загрузились в БД из файла истории (БЕЗ маппинга)
raw_cf = mart.get_history_cash_flow(canonical=False)
print("📥 RAW данные из БД (до маппинга):")
print(f"   Загружено: {raw_cf.shape[0]} строк, {raw_cf.shape[1]} колонок")

# CANONICAL данные - после применения маппинга в DataMart
canonical_cf = mart.get_history_cash_flow(canonical=True)
print(f"\n📤 CANONICAL данные из БД (после маппинга):")
print(f"   Загружено: {canonical_cf.shape[0]} строк, {canonical_cf.shape[1]} колонок")

# Получаем данные 2024 из RAW формата
raw_2024 = {}
if not raw_cf.empty and 'metric' in raw_cf.columns:
    if '2024' in raw_cf.columns:
        for _, row in raw_cf.iterrows():
            metric = row['metric']
            value = pd.to_numeric(row['2024'], errors='coerce')
            if pd.notna(value):
                raw_2024[metric] = float(value)
    print(f"\n✅ Загружено {len(raw_2024)} статей из RAW формата для 2024 года")

# Получаем данные 2024 из canonical формата
canonical_2024 = {}
if not canonical_cf.empty and 'metric' in canonical_cf.columns:
    if '2024' in canonical_cf.columns:
        for _, row in canonical_cf.iterrows():
            metric = row['metric']
            value = pd.to_numeric(row['2024'], errors='coerce')
            if pd.notna(value):
                canonical_2024[metric] = float(value)
    print(f"\n✅ Загружено {len(canonical_2024)} статей из canonical формата для 2024 года")

mart.close()

## 🔗 ШАГ 4: Загрузка маппинга статей

**Примечание**: Маппинг статей из официального отчета на canonical метрики происходит автоматически в препроцессинге при загрузке данных в БД через DataMart.

In [ ]:
# === Загрузка конфигурации маппинга из excel_loader.yaml ===

excel_loader_path = croot / "configs" / "excel_loader.yaml"
required_metrics = {}

if excel_loader_path.exists():
    with open(excel_loader_path, 'r', encoding='utf-8') as f:
        mapping_config = yaml.safe_load(f)
    
    cf_mapping = mapping_config.get('history', {}).get('CF', {})
    if cf_mapping:
        required_metrics = cf_mapping.get('required_metrics', {})
        print(f"✅ Найдено {len(required_metrics)} метрик в маппинге CF")
    else:
        print("⚠️ Конфигурация CF не найдена в excel_loader.yaml")
else:
    print(f"❌ Файл не найден: {excel_loader_path}")

## 🔍 ШАГ 5: Детальное сравнение всех источников

**Цель**: Сравнить данные из официального отчета с данными в БД на разных этапах обработки.

In [ ]:
# === Детальное сравнение всех источников ===

def similarity(a, b):
    """Вычисляет схожесть между двумя строками"""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def normalize_name(name):
    """Нормализует название метрики для сравнения"""
    name = name.lower()
    name = re.sub(r'[^a-z0-9]', '', name)
    return name

def normalize_name_simple(name):
    """Простая нормализация названия метрики"""
    return name.lower().strip()

def find_best_match(name, candidates, threshold=0.6):
    """Находит лучшее совпадение для названия среди кандидатов"""
    if not candidates:
        return None, 0.0
    
    best_match = None
    best_score = 0.0
    
    normalized_name = normalize_name(name)
    
    for candidate in candidates:
        normalized_candidate = normalize_name(candidate)
        score = similarity(normalized_name, normalized_candidate)
        
        if score > best_score:
            best_score = score
            best_match = candidate
    
    if best_score >= threshold:
        return best_match, best_score
    
    return None, best_score

# Создаем маппинг официальных названий на canonical метрики
official_to_canonical_map = {
    'Net earnings': 'net_income',
    'Depreciation, depletion and amortization': 'total_da',
    'Asset impairment charges': 'asset_impairment_charges_cf',
    'Gain on equity investee transactions': 'gain_on_equity_investee_transactions_cf',
    'Restructuring and other charges': 'restructuring_and_other_charges_cf',
    'Loss on debt extinguishment': 'loss_on_debt_extinguishment_cf',
    'Pensions and other post-employment benefits': 'pensions_and_other_post_employment_benefits',
    'Active employee benefit investments': 'active_employee_benefit_investments',
    'Deferred income taxes': 'tax_deferred_adjustment',
    'Net gain on sale of assets': 'gain_loss_reversal',
    'Equity investees earnings, net of distributions received': 'equity_investees_earnings_net_of_distributions',
    'Changes in: Current receivables': 'wc_accounts_receivable_change',
    'Inventories': 'wc_inventory_change',
    'Current accounts payable and accrued expenses': 'wc_accounts_payable_change',
    'Income taxes receivable/payable': 'wc_income_taxes_receivable_payable_change',
    'All other, net': 'wc_all_other_net',
    'Net cash provided by operating activities': 'cfo',
    'Capital expenditures': 'capex',
    'Proceeds from cost reimbursement government grants': 'proceeds_from_cost_reimbursement_government_grants',
    'Proceeds from sale of assets': 'disposal_proceeds',
    'Proceeds from sale of ownership interests in equity investees': 'proceeds_from_sale_of_ownership_interests_in_equity_investees',
    'Other investing activities': 'cfi_other',
    'Net cash used in investing activities': 'cfi',
    'Issuance of long-term debt, net of financing costs': 'cff_borrowings',
    'Repayment of long-term debt': 'cff_repayments',
    'Common stock repurchased': 'cff_share_repurchases',
    'Proceeds from government incentives': 'proceeds_from_government_incentives',
    'Other financing activities': 'cff_other',
    'Net cash used in financing activities': 'cff',
    'Effect of exchange rate changes on cash': 'cf_fx_effect',
    'Net (decrease) increase in cash, cash equivalents and restricted cash': 'net_change',
    'Cash, cash equivalents and restricted cash at beginning of year': 'cash_opening',
    'Cash, cash equivalents and restricted cash at end of year': 'cash_ending',
}

print("="*80)
print("🔍 Детальное сравнение данных 2024")
print("="*80)

comparison_data = []

for official_name, official_value in official_reporting_2024.items():
    canonical_metric = official_to_canonical_map.get(official_name)
    
    if canonical_metric:
        canonical_value = canonical_2024.get(canonical_metric)
        raw_value = raw_2024.get(canonical_metric)
        
        if canonical_value is not None:
            diff = abs(official_value - canonical_value)
            status = '✅' if diff < 1000 else '⚠️'
            comparison_data.append({
                'Официальное название': official_name,
                'Canonical метрика': canonical_metric,
                'Офиц. значение': f"${official_value/1_000_000:,.0f}M",
                'Canonical значение': f"${canonical_value/1_000_000:,.0f}M",
                'Разница': f"${diff/1_000_000:,.2f}M",
                'Статус': status
            })
        else:
            comparison_data.append({
                'Официальное название': official_name,
                'Canonical метрика': canonical_metric,
                'Офиц. значение': f"${official_value/1_000_000:,.0f}M",
                'Canonical значение': 'НЕ НАЙДЕНО',
                'Разница': '-',
                'Статус': '❌'
            })
    else:
        comparison_data.append({
            'Официальное название': official_name,
            'Canonical метрика': 'НЕ МАППИТСЯ',
            'Офиц. значение': f"${official_value/1_000_000:,.0f}M",
            'Canonical значение': '-',
            'Разница': '-',
            'Статус': '⚠️'
        })

comparison_df = pd.DataFrame(comparison_data)
display(Markdown("### 📊 Детальное сравнение данных 2024"))
display(comparison_df.style.set_table_styles([
    {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
]).set_properties(**{'font-size': '9px'}))

print(f"\n📋 Всего статей: {len(comparison_data)}")
print(f"   ✅ Совпадает: {len([x for x in comparison_data if '✅' in x['Статус']])}")
print(f"   ⚠️ Расхождения: {len([x for x in comparison_data if '⚠️' in x['Статус']])}")
print(f"   ❌ Не найдено: {len([x for x in comparison_data if '❌' in x['Статус']])}")

## 📋 ШАГ 6: План исправлений на основе анализа

**Цель**: Создать детальный план исправлений для всех выявленных проблем в процессе загрузки данных CF

**Задачи**:
1. Проанализировать расхождения между официальными данными и данными в БД
2. Определить причины расхождений (маппинг, суммирование, отсутствие данных)
3. Создать план исправлений для каждой метрики
4. Определить порядок применения исправлений


In [ ]:
# === ШАГ 6: План исправлений на основе анализа ===

display(Markdown("### 📋 План исправлений на основе анализа"))

print("="*80)
print("📋 ШАГ 6: ПЛАН ИСПРАВЛЕНИЙ")
print("="*80)

# Анализируем расхождения из ШАГ 5
if 'comparison_data' not in globals():
    print("⚠️ Сначала выполните ШАГ 5 для получения данных сравнения")
else:
    # Группируем проблемы по типам
    problems_by_type = {
        'Не найдено': [],
        'Расхождения': [],
        'Не маппится': []
    }
    
    for item in comparison_data:
        status = item.get('Статус', '')
        if '❌' in status:
            problems_by_type['Не найдено'].append(item)
        elif '⚠️' in status:
            if item.get('Canonical метрика') == 'НЕ МАППИТСЯ':
                problems_by_type['Не маппится'].append(item)
            else:
                problems_by_type['Расхождения'].append(item)
    
    print(f"\n📊 Статистика проблем:")
    print(f"   ❌ Не найдено: {len(problems_by_type['Не найдено'])}")
    print(f"   ⚠️ Расхождения: {len(problems_by_type['Расхождения'])}")
    print(f"   ⚠️ Не маппится: {len(problems_by_type['Не маппится'])}")
    
    # Создаем план исправлений
    print(f"\n📋 ПЛАН ИСПРАВЛЕНИЙ:")
    print(f"\n1️⃣ Метрики, которые не найдены в БД:")
    for item in problems_by_type['Не найдено'][:10]:
        print(f"   • {item.get('Canonical метрика')}: добавить в БД и маппинг")
    
    print(f"\n2️⃣ Метрики с расхождениями:")
    for item in problems_by_type['Расхождения'][:10]:
        print(f"   • {item.get('Canonical метрика')}: проверить маппинг и суммирование")
    
    print(f"\n3️⃣ Статьи, которые не маппятся:")
    for item in problems_by_type['Не маппится'][:10]:
        print(f"   • {item.get('Официальное название')}: добавить маппинг в excel_loader.yaml")
    
    print(f"\n✅ План создан. Переходим к ШАГ 7 для создания исправленного canonical формата.")


## 🔧 ШАГ 7: Создание правильного canonical формата

**Цель**: Создать исправленный canonical формат для 2024 года на основе официальных данных и анализа расхождений

**Процесс**:
1. Использовать официальные данные как источник истины
2. Применить правильный маппинг из `official_to_canonical_map`
3. Суммировать статьи, которые должны быть объединены
4. Создать исправленный canonical формат для всех метрик CF


In [ ]:
# === Создание правильного canonical формата на основе официальной отчетности ===

display(Markdown("### 🔧 Создание правильного canonical формата для 2024 года"))

if 'official_reporting_2024' not in globals() or not official_reporting_2024:
    print("❌ Данные из официальной отчетности не загружены. Запустите ячейку ШАГ 1.")
elif 'canonical_2024' not in globals() or not canonical_2024:
    print("❌ Данные из canonical формата не загружены. Запустите ячейку ШАГ 3.")
else:
    # Создаём маппинг алиасов из excel_loader.yaml
    aliases_map = {}
    if 'required_metrics' in globals() and required_metrics:
        for metric, config in required_metrics.items():
            aliases = config.get('aliases', [])
            for alias in aliases:
                aliases_map[alias.lower()] = metric
            aliases_map[metric.lower()] = metric
    
    # Используем правильный маппинг из ШАГ 5, если он создан
    if 'official_to_canonical_map' in globals() and official_to_canonical_map:
        print("✅ Используем правильный маппинг из ШАГ 5")
        print(f"   Загружено {len(official_to_canonical_map)} правильных сопоставлений")
        mapping_to_use = official_to_canonical_map
    else:
        print("⚠️ Правильный маппинг не найден, используем автоматический поиск")
        mapping_to_use = {}
    
    # Создаём функцию для поиска canonical метрики
    def find_canonical_metric_for_step7(official_name):
        """Находит canonical метрику используя многоуровневую систему поиска"""
        # ПРИОРИТЕТ 1: Используем явный маппинг официальных названий (из ШАГ 5)
        if official_name in mapping_to_use:
            canonical_name = mapping_to_use[official_name]
            if canonical_name in canonical_2024 or canonical_name in CANONICAL_METRICS_STANDARD.get('CF', []):
                return canonical_name, 'official_to_canonical_map'
        
        # ПРИОРИТЕТ 2: Ищем через алиасы (нормализуем название)
        official_normalized = normalize_name(official_name)
        official_normalized_no_spaces = official_normalized.replace(' ', '')
        
        if official_normalized in aliases_map:
            canonical_name = aliases_map[official_normalized]
            if canonical_name in canonical_2024 or canonical_name in CANONICAL_METRICS_STANDARD.get('CF', []):
                return canonical_name, 'aliases_map'
        elif official_normalized_no_spaces in aliases_map:
            canonical_name = aliases_map[official_normalized_no_spaces]
            if canonical_name in canonical_2024 or canonical_name in CANONICAL_METRICS_STANDARD.get('CF', []):
                return canonical_name, 'aliases_map'
        else:
            # Проверяем частичное совпадение с алиасами
            for alias, canonical in aliases_map.items():
                alias_normalized = normalize_name_simple(alias).replace(' ', '')
                if (official_normalized in alias or alias in official_normalized or
                    official_normalized_no_spaces in alias_normalized or alias_normalized in official_normalized_no_spaces):
                    if canonical in canonical_2024 or canonical in CANONICAL_METRICS_STANDARD.get('CF', []):
                        return canonical, 'aliases_map_partial'
        
        # ПРИОРИТЕТ 3: По схожести с canonical метриками
        from engine.database.data_mart import CANONICAL_METRICS_STANDARD
        cf_metrics = CANONICAL_METRICS_STANDARD.get('CF', [])
        canonical_match, score = find_best_match(official_name, cf_metrics, threshold=0.6)
        if canonical_match:
            return canonical_match, f'similarity ({score:.1%})'
        
        return None, None
    
    # Создаём правильный canonical формат
    corrected_canonical = {}
    mapping_log = []
    
    print("\n📋 Создание маппинга официальных статей на canonical метрики:")
    
    # Словарь для отслеживания статей, которые должны суммироваться
    articles_to_sum = {}  # {canonical_metric: [list of official items]}
    
    for official_item, official_val in official_reporting_2024.items():
        # Используем многоуровневую систему поиска
        canonical_metric, method = find_canonical_metric_for_step7(official_item)
        
        if canonical_metric:
            # ВАЖНО: Если несколько статей маппятся в одну canonical метрику, они должны СУММИРОВАТЬСЯ
            if canonical_metric not in articles_to_sum:
                articles_to_sum[canonical_metric] = []
            articles_to_sum[canonical_metric].append({
                'official_item': official_item,
                'value': official_val,
                'method': method
            })
        else:
            # Статья не маппится - должна попасть в "прочее" (для CF обычно это cfo_other, cfi_other, cff_other)
            # Определяем тип на основе названия
            item_lower = official_item.lower()
            if 'operating' in item_lower or 'cfo' in item_lower:
                other_metric = "cfo_other"
            elif 'investing' in item_lower or 'cfi' in item_lower:
                other_metric = "cfi_other"
            elif 'financing' in item_lower or 'cff' in item_lower:
                other_metric = "cff_other"
            else:
                # По умолчанию в cfo_other
                other_metric = "cfo_other"
            
            # Добавляем в суммирование для "прочего"
            if other_metric not in articles_to_sum:
                articles_to_sum[other_metric] = []
            articles_to_sum[other_metric].append({
                'official_item': official_item,
                'value': official_val,
                'method': 'other_items'
            })
            
            print(f"⚠️ Статья '{official_item}' не сопоставлена, будет добавлена в '{other_metric}'")
    
    # Теперь суммируем все статьи для каждой canonical метрики
    print("\n📊 Суммирование статей для canonical метрик:")
    for canonical_metric, articles_list in articles_to_sum.items():
        if len(articles_list) > 1:
            # ВАЖНО: Проверяем, есть ли среди статей "Total" или агрегированная статья
            # Если есть, используем её вместо суммирования компонентов
            total_articles = [art for art in articles_list if 'total' in art['official_item'].lower() or 'net cash' in art['official_item'].lower()]
            
            # ВАЖНО: Исключаем агрегированные статьи, которые нельзя суммировать
            aggregated_articles = [
                'net cash provided by operating activities',
                'net cash used in investing activities',
                'net cash used in financing activities',
                'net (decrease) increase in cash'
            ]
            
            # Фильтруем агрегированные статьи
            filtered_articles = []
            excluded_aggregated = []
            for art in articles_list:
                art_lower = art['official_item'].lower()
                is_aggregated = any(agg in art_lower for agg in aggregated_articles)
                if is_aggregated:
                    excluded_aggregated.append(art)
                else:
                    filtered_articles.append(art)
            
            if excluded_aggregated:
                print(f"     ⚠️ Исключены агрегированные статьи (нельзя суммировать с компонентами):")
                for art in excluded_aggregated:
                    print(f"       - {art['official_item']}: ${art['value']:,.0f}")
            
            # Используем отфильтрованный список
            articles_list = filtered_articles
            
            if total_articles and len(articles_list) > 1:
                # ВАЖНО: Используем агрегированную статью (например, "Net cash provided by operating activities")
                # ИСКЛЮЧАЕМ компоненты из суммирования, т.к. они уже учтены в Total
                total_art = total_articles[0]
                total_value = total_art['value']
                
                print(f"  ✅ {canonical_metric}: используется агрегированная статья '{total_art['official_item']}' = ${total_value:,.0f}")
                print(f"     (исключены компоненты из суммирования, т.к. 'Total' уже содержит сумму):")
                for art in articles_list:
                    if art != total_art:
                        print(f"       - {art['official_item']}: ${art['value']:,.0f} (компонент, учтен в Total)")
                
                # Добавляем только агрегированную статью в mapping_log
                mapping_log.append({
                    'Официальное название': total_art['official_item'],
                    'Canonical метрика': canonical_metric,
                    'Метод': total_art['method'],
                    'Офиц. значение': f"${total_art['value']:,.0f}",
                    'Текущее в БД': f"${canonical_2024.get(canonical_metric, 0):,.0f}" if canonical_2024 else '-',
                    'Статус': 'используется агрегированная'
                })
                
                corrected_canonical[canonical_metric] = total_value
            else:
                # Несколько статей суммируются (нет агрегированной)
                total_value = sum(art['value'] for art in articles_list)
                print(f"  ✅ {canonical_metric}: суммируется из {len(articles_list)} статей = ${total_value:,.0f}")
                for art in articles_list:
                    print(f"     - {art['official_item']}: ${art['value']:,.0f} (via {art['method']})")
                    mapping_log.append({
                        'Официальное название': art['official_item'],
                        'Canonical метрика': canonical_metric,
                        'Метод': art['method'],
                        'Офиц. значение': f"${art['value']:,.0f}",
                        'Текущее в БД': f"${canonical_2024.get(canonical_metric, 0):,.0f}" if canonical_2024 else '-',
                        'Статус': 'суммируется'
                    })
                
                corrected_canonical[canonical_metric] = total_value
        else:
            # Одна статья - просто присваиваем значение
            art = articles_list[0]
            corrected_canonical[canonical_metric] = art['value']
            mapping_log.append({
                'Официальное название': art['official_item'],
                'Canonical метрика': canonical_metric,
                'Метод': art['method'],
                'Офиц. значение': f"${art['value']:,.0f}",
                'Текущее в БД': f"${canonical_2024.get(canonical_metric, 0):,.0f}" if canonical_2024 else '-',
                'Статус': 'обновлено' if canonical_metric in canonical_2024 else 'новая метрика'
            })
        
        # Проверяем, есть ли эта метрика в текущем canonical формате
        current_canonical_val = canonical_2024.get(canonical_metric)
        if current_canonical_val is not None:
            final_value = corrected_canonical[canonical_metric]
            diff = final_value - current_canonical_val
            if abs(diff) > 1000:  # Разница больше $1K
                status_icon = '✅'
                status_text = 'обновлено'
                print(f"{status_icon} {canonical_metric}: ${final_value:,.0f} - {status_text} (было ${current_canonical_val:,.0f}, diff=${diff:,.0f})")
            else:
                print(f"✅ {canonical_metric}: ${final_value:,.0f} - совпадает")
        else:
            print(f"🆕 {canonical_metric}: ${corrected_canonical[canonical_metric]:,.0f} - новая метрика")
    
    # Добавляем статьи, которые есть в canonical, но не в официальном отчете (оставляем как есть)
    for canonical_metric, canonical_val in canonical_2024.items():
        if canonical_metric not in corrected_canonical:
            corrected_canonical[canonical_metric] = canonical_val
            print(f"ℹ️ {canonical_metric}: ${canonical_val:,.0f} (сохранено из canonical)")
    
    print(f"\n✅ Создан исправленный canonical формат: {len(corrected_canonical)} статей")
    
    # Показываем маппинг
    if mapping_log:
        mapping_df = pd.DataFrame(mapping_log)
        display(Markdown("#### 📋 Таблица маппинга:"))
        display(mapping_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
    
    # Показываем расхождения
    print("\n" + "="*80)
    print("Расхождения между оригинальным и исправленным canonical:")
    
    differences = []
    for metric in sorted(set(canonical_2024.keys()) | set(corrected_canonical.keys())):
        orig_val = canonical_2024.get(metric, 0)
        corr_val = corrected_canonical.get(metric, 0)
        
        if abs(orig_val - corr_val) > 1000:  # Разница больше $1K
            diff = corr_val - orig_val
            diff_pct = (diff / abs(orig_val) * 100) if orig_val != 0 else 0
            differences.append({
                'Метрика': metric,
                'Оригинал (canonical)': f"${orig_val:,.0f}",
                'Исправлено (офиц.)': f"${corr_val:,.0f}",
                'Разница': f"${diff:,.0f}",
                'Разница %': f"{diff_pct:+.1f}%"
            })
    
    if differences:
        diff_df = pd.DataFrame(differences)
        display(Markdown("#### ❌ Статьи с расхождениями:"))
        display(diff_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
        print(f"\n⚠️ Обнаружено {len(differences)} статей с расхождениями > $1K")
    else:
        print("✅ Расхождений не найдено")
    
    # Сохраняем в переменную
    corrected_canonical_2024 = corrected_canonical
    print("\n✅ Исправленный canonical формат сохранен в переменную 'corrected_canonical_2024'")


## 🔧 ШАГ 7.1: Применение исправлений

**Цель**: Применить все исправления на основе правильного canonical формата:
1. Обновить CSV файл истории для 2024 года
2. Обновить данные в БД через `save_history_records`
3. Проверить, что маппинг RAW → CANONICAL работает правильно
4. Убедиться, что данные правильно загружаются в препроцессинг


In [ ]:
# === Применение исправлений ===

display(Markdown("### 🔧 Применение исправлений"))

if 'corrected_canonical_2024' not in globals() or not corrected_canonical_2024:
    print("❌ Исправленный canonical формат не создан. Запустите ШАГ 7.")
    print("   ШАГ 7 создаст переменную corrected_canonical_2024 с правильными значениями")
else:
    csv_updated = False
    db_updated = False
    print("="*80)
    print("🔧 ПРИМЕНЕНИЕ ИСПРАВЛЕНИЙ")
    print("="*80)
    
    # 1. Обновление CSV файла истории
    print(f"\n📝 ШАГ 1: Обновление CSV файла истории для 2024 года")
    
    csv_path = croot / "history" / "cf_history_us_steel.csv"
    
    if not csv_path.exists():
        print(f"⚠️ CSV файл не найден: {csv_path}")
        print(f"   Создаем новый CSV файл...")
        # Создаем новый CSV файл
        df_csv = pd.DataFrame({
            'metric': list(corrected_canonical_2024.keys()),
            '2024': list(corrected_canonical_2024.values())
        })
        df_csv.to_csv(csv_path, index=False)
        print(f"   ✅ Создан новый CSV файл: {csv_path}")
        csv_updated = True
    else:
        df_csv = pd.read_csv(csv_path)
        
        # Обновляем значения для 2024 года
        updates_count = 0
        new_metrics_count = 0
        
        # ВАЖНО: corrected_canonical_2024 уже содержит правильные суммированные значения из ШАГ 7
        # Применяем их как есть, без дополнительного суммирования
        for metric, value in corrected_canonical_2024.items():
            if metric in df_csv['metric'].values:
                # Обновляем существующую метрику значением из corrected_canonical_2024 (уже суммировано)
                idx = df_csv[df_csv['metric'] == metric].index[0]
                old_value = pd.to_numeric(df_csv.at[idx, '2024'], errors='coerce')
                if pd.isna(old_value):
                    old_value = 0.0
                df_csv.at[idx, '2024'] = value
                if abs(old_value - value) > 1000:
                    updates_count += 1
                    print(f"   ✅ Обновлено {metric}: ${old_value/1_000_000:,.0f}M → ${value/1_000_000:,.0f}M")
            else:
                # Добавляем новую метрику
                new_row = {'metric': metric}
                # Заполняем нулями для всех годов кроме 2024
                for col in df_csv.columns:
                    if col == 'metric':
                        continue
                    new_row[col] = 0.0
                new_row['2024'] = value
                df_csv = pd.concat([df_csv, pd.DataFrame([new_row])], ignore_index=True)
                new_metrics_count += 1
                print(f"   🆕 Добавлено {metric}: ${value/1_000_000:,.0f}M")
        
        # Сохраняем обновленный CSV
        df_csv.to_csv(csv_path, index=False)
        print(f"\n✅ CSV файл обновлен:")
        print(f"   Обновлено метрик: {updates_count}")
        print(f"   Добавлено новых метрик: {new_metrics_count}")
        print(f"   Путь: {csv_path}")
        
        csv_updated = True
    
    # 2. Обновление данных в БД
    print(f"\n📝 ШАГ 2: Обновление данных в БД для 2024 года")
    
    mart = get_data_mart(ROOT, COMPANY)
    
    # Подготавливаем записи для БД
    records = []
    for metric, value in corrected_canonical_2024.items():
        records.append({
            'metric': metric,
            'year': 2024,
            'value': float(value)
        })
    
    try:
        # Обновляем данные в БД через save_history_records
        mart.save_history_records(
            statement_type='CF',
            records=records,
            replace_existing=True
        )
        
        print(f"✅ Данные обновлены в БД:")
        print(f"   Обновлено метрик: {len(records)}")
        print(f"   Год: 2024")
        
        db_updated = True
        
        # Проверяем обновленные данные
        print(f"\n🔍 Проверка обновленных данных в БД:")
        cf_canonical_updated = mart.get_history_cash_flow(canonical=True)
        
        if not cf_canonical_updated.empty and '2024' in cf_canonical_updated.columns:
            verified_count = 0
            mismatches = []
            
            for metric, expected_value in corrected_canonical_2024.items():
                metric_row = cf_canonical_updated[cf_canonical_updated['metric'].str.lower() == metric.lower()]
                if not metric_row.empty:
                    actual_value = pd.to_numeric(metric_row.iloc[0]['2024'], errors='coerce')
                    if pd.notna(actual_value):
                        diff = abs(actual_value - expected_value)
                        if diff < 1000:  # Допустимая погрешность
                            verified_count += 1
                        else:
                            mismatches.append({
                                'metric': metric,
                                'expected': expected_value,
                                'actual': actual_value,
                                'diff': diff
                            })
            
            print(f"   ✅ Проверено метрик: {verified_count}/{len(corrected_canonical_2024)}")
            
            if mismatches:
                print(f"   ⚠️ Найдено расхождений: {len(mismatches)}")
                for mm in mismatches[:5]:  # Показываем первые 5
                    print(f"      • {mm['metric']}: ожидается ${mm['expected']/1_000_000:,.0f}M, в БД ${mm['actual']/1_000_000:,.0f}M (разница: ${mm['diff']/1_000_000:,.0f}M)")
            
            if verified_count == len(corrected_canonical_2024):
                print(f"   ✅ Все метрики успешно обновлены и проверены!")
        
    except Exception as e:
        print(f"❌ Ошибка при обновлении БД: {e}")
        import traceback
        traceback.print_exc()
        db_updated = False
    
    finally:
        mart.close()
    
    # Итоговая сводка
    print(f"\n" + "="*80)
    print("📊 ИТОГОВАЯ СВОДКА")
    print("="*80)
    print(f"   CSV файл: {'✅ Обновлен' if csv_updated else '❌ Не обновлен'}")
    print(f"   БД: {'✅ Обновлена' if db_updated else '❌ Не обновлена'}")
    
    if csv_updated and db_updated:
        print(f"\n✅ Все исправления успешно применены!")
        print(f"   Данные для 2024 года обновлены в CSV файле и БД")
    else:
        print(f"\n⚠️ Некоторые исправления не применены. Проверьте ошибки выше.")
